In [ ]:
import os
import sys
import numpy as np
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from packages import JointInferenceProcedure

datasets =  [dict(filename="data/linear_viscoelastic/angle45/t0-0p4.out", t_unload=0.4, force = 8 * np.pi, theta = 45),
            dict(filename="data/linear_viscoelastic/angle0/delta-0.1-0.out", t_unload=0.2, force=8 * np.pi, theta=0)]

model = JointInferenceProcedure(
    force=8 * np.pi,
    a=1.0,
    theta=45,
    material_model="viscoelastic",
    boundary_model="bounded",
    sigma_noise_percent=0.1,
    sigma_bias=None,        
    eta_s_bounds=(0.01, 50.0),
    eta_p_bounds=(0.01, 50.0),
    lambda_bounds=(0.01, 50.0),
    delta0_bounds=(0.01, 50.0),
    nsteps=6000,
    nwalkers=16,
    thin_factor=2)

model.load_datasets(datasets)
theta_true = [0.5, 0.9, 0.1, 0.1]   # eta_s, eta_p, lambda_, delta0

In [ ]:
for i, spec in enumerate(datasets):
    print(f"{os.path.basename(spec['filename'])}  t_unload={spec['t_unload']}  "
          f"{spec.get('boundary_model', 'bounded')}")
    model.select(i).plot_data(theta_true=theta_true)

In [ ]:
samples = model.run_mcmc(warmup=True)

In [ ]:
names = model._get_parameter_labels(latex=False)
print(f"{'parameter':<14}{'median':>12}{'-1 sigma':>12}{'+1 sigma':>12}{'truth':>10}")
for i, name in enumerate(names):
    lo, mid, hi = np.percentile(samples[:, i], [16, 50, 84])
    truth = f"{theta_true[i]:.4f}" if i < len(theta_true) else ""
    print(f"{name:<14}{mid:>12.4f}{mid - lo:>12.4f}{hi - mid:>12.4f}{truth:>10}")

In [ ]:
model.plot_corner(theta_true)
model.plot_trace()
for i, spec in enumerate(datasets):
    print(f"{os.path.basename(spec['filename'])}  t_unload={spec['t_unload']}")
    model.select(i).plot_posterior_predictive()